In [3]:
!pip install -q transformers datasets sentencepiece accelerate

In [4]:
from transformers import VisionEncoderDecoderModel, DonutProcessor
import torch

# This is the class the previous error was referring to
print("✅ VisionEncoderDecoderModel successfully imported from Transformers!")

# Quick check on your 10-core GPU
if torch.backends.mps.is_available():
    print("🚀 M4 GPU (MPS) is ready for training.")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ VisionEncoderDecoderModel successfully imported from Transformers!
🚀 M4 GPU (MPS) is ready for training.


In [6]:
import json
from transformers import DonutProcessor

# 1. Re-define the processor (Ensure this matches your model choice)
model_checkpoint = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(model_checkpoint)

# 2. The Collator Class
class DonutDataCollator:
    def __init__(self, processor, max_length=128):
        self.processor = processor
        self.max_length = max_length

    def __call__(self, batch):
        pixel_values = [item["image"].convert("RGB") for item in batch]
        inputs = self.processor(pixel_values, return_tensors="pt")
        
        # Extract ground truth from the JSONL metadata
        labels = [json.loads(item["ground_truth"])["gt_parse"] for item in batch]
        
        labels_batch = self.processor.tokenizer(
            labels, 
            add_special_tokens=True, 
            max_length=self.max_length, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )

        labels = labels_batch.input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {
            "pixel_values": inputs.pixel_values,
            "labels": labels
        }

# 3. Now initialize - This should work perfectly now!
collator = DonutDataCollator(processor)
print("✅ Collator initialized successfully with the processor!")

The image processor of type `DonutImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ Collator initialized successfully with the processor!


In [7]:
import os
import shutil
import json

# Paths
ORIGINAL_DIR = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset/vbar_categorical_plots/train"
SAMPLE_DIR = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset/vbar_sample"

os.makedirs(os.path.join(SAMPLE_DIR, "png"), exist_ok=True)

# Copy 500 images
images = os.listdir(os.path.join(ORIGINAL_DIR, "png"))[:500]
for img in images:
    shutil.copy(os.path.join(ORIGINAL_DIR, "png", img), os.path.join(SAMPLE_DIR, "png", img))

# Filter metadata.jsonl for only these 500 images
sample_metadata = []
with open(os.path.join(ORIGINAL_DIR, "metadata.jsonl"), 'r') as f:
    for line in f:
        data = json.loads(line)
        if data['file_name'] in images:
            sample_metadata.append(data)

with open(os.path.join(SAMPLE_DIR, "metadata.jsonl"), 'w') as f:
    for entry in sample_metadata:
        f.write(json.dumps(entry) + '\n')

print(f"✅ Created sample dataset with {len(images)} images at {SAMPLE_DIR}")

✅ Created sample dataset with 500 images at /Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset/vbar_sample


NameError: name 'dataset' is not defined

In [10]:
from transformers import VisionEncoderDecoderModel, DonutProcessor
import torch

# 1. Setup Device
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 2. Load Model and Processor
model_checkpoint = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(model_checkpoint)
model = VisionEncoderDecoderModel.from_pretrained(model_checkpoint)

# 3. Move model to your M4 GPU
model.to(device)

print(f"✅ Model 'model' is now defined and running on: {device}")

Loading weights: 100%|█| 484/484 [00:00<00:00, 1864.73it/s, Materializing param=encoder.encoder.layers.3.blocks.
The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model 'model' is now defined and running on: mps


In [12]:
import os
import torch
from datasets import load_dataset
from transformers import VisionEncoderDecoderModel, DonutProcessor

# 1. Paths & Device
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset"
DATA_DIR = os.path.join(BASE_PATH, "vbar_categorical_plots")
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 2. Load Dataset
print("📂 Loading dataset from local SSD...")
dataset = load_dataset("imagefolder", data_dir=DATA_DIR)

# 3. Load Model and Processor
print("🤖 Loading Donut model to M4 GPU...")
model_checkpoint = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(model_checkpoint)
model = VisionEncoderDecoderModel.from_pretrained(model_checkpoint).to(device)

print(f"✅ 'dataset' and 'model' are now live on: {device}")

📂 Loading dataset from local SSD...


Generating train split: 52463 examples [00:00, 125679.18 examples/s]
Generating validation split: 11240 examples [00:00, 127820.00 examples/s]


🤖 Loading Donut model to M4 GPU...


Loading weights: 100%|█| 484/484 [00:00<00:00, 1908.25it/s, Materializing param=encoder.encoder.layers.3.blocks.
The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ 'dataset' and 'model' are now live on: mps


In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./stem_sight_sample_run",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=100,                        # Just run 100 steps to test the logic
    learning_rate=5e-5,
    logging_steps=10,                     # Log every 10 steps to see progress
    save_total_limit=1,
    #use_mps_device=True,                  # Use your 10-core M4 GPU
    remove_unused_columns=False,
    report_to="none"                      # Keep it clean
)

# Initialize Trainer with the sample data
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].select(range(500)), # Only use the 500 sample images
    data_collator=collator,
)

print("Running sample training on M4...")
trainer.train()

Running sample training on M4...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset/vbar_categorical_plots/train/104649.png'

In [14]:
import os
from datasets import load_dataset

# Point this EXACTLY to the folder containing your 'train' and 'validation' sub-folders
DATA_DIR = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset/vbar_categorical_plots"

print("🔄 Re-syncing dataset paths...")
# This will look into vbar_categorical_plots/train/png/ automatically
dataset = load_dataset("imagefolder", data_dir=DATA_DIR)

print(f"✅ Paths synced. Found {len(dataset['train'])} training images.")

🔄 Re-syncing dataset paths...
✅ Paths synced. Found 52463 training images.


In [23]:
import torch
import gc
import os
from transformers import TrainingArguments, Trainer

# 1. Clear memory & setup device
gc.collect()
torch.mps.empty_cache()
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 2. Hard-limit the image size
processor.image_processor.size = {"height": 480, "width": 640}

# 3. Freeze the Encoder (Swin Transformer)
model.encoder.requires_grad_(False)
print("❄️ Encoder frozen. GPU focus is now on the Decoder.")

# 4. Ultra-conservative Training Args (Cleaned)
training_args = TrainingArguments(
    output_dir="./stem_sight_ultra_light",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,       
    max_steps=50,                         
    learning_rate=1e-4,                   
    logging_steps=5,
    gradient_checkpointing=True,  
    remove_unused_columns=False,
    report_to="none"
)

# 5. Environment variable check
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].select(range(100)), 
    data_collator=collator,
)

print("🚀 Launching Ultra-Light training on M4...")
trainer.train()

❄️ Encoder frozen. GPU focus is now on the Decoder.
🚀 Launching Ultra-Light training on M4...


AttributeError: 'VisionEncoderDecoderConfig' object has no attribute 'pad_token_id'

In [18]:
import os
from datasets import load_dataset

# Path to the specialized folder
DATA_DIR = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset/vbar_categorical_plots"

print("🔄 Force-refreshing dataset and clearing path cache...")

# We use split="train" to verify immediately if it can see the images
dataset = load_dataset(
    "imagefolder", 
    data_dir=DATA_DIR,
    drop_labels=True, # We use the JSONL for labels, not folder names
    download_mode="force_redownload" 
)

# Test a single image access immediately
try:
    test_img = dataset["train"][0]["image"]
    print(f"✅ Success! Found {len(dataset['train'])} images. First image loaded correctly.")
except Exception as e:
    print(f"❌ Still failing. Error: {e}")

🔄 Force-refreshing dataset and clearing path cache...


Generating train split: 52463 examples [00:00, 124120.49 examples/s]
Generating validation split: 11240 examples [00:00, 125642.89 examples/s]


✅ Success! Found 52463 images. First image loaded correctly.


In [24]:
# 1. Manually sync the token IDs from the decoder to the main config
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_chartqa>'])[0]

# 2. Double check they are set
print(f"Pad Token ID: {model.config.pad_token_id}")
print(f"Decoder Start Token ID: {model.config.decoder_start_token_id}")

Pad Token ID: 1
Decoder Start Token ID: 3


In [25]:
# 1. Manually sync the token IDs from the decoder to the main config
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_chartqa>'])[0]

# 2. Double check they are set
print(f"Pad Token ID: {model.config.pad_token_id}")
print(f"Decoder Start Token ID: {model.config.decoder_start_token_id}")

Pad Token ID: 1
Decoder Start Token ID: 3


In [32]:
import os
import json
import torch
import gc
from datasets import load_dataset
from transformers import (
    DonutProcessor, 
    VisionEncoderDecoderModel, 
    TrainingArguments, 
    Trainer
)

# 1. HARDWARE & MEMORY SETUP
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0" 
device = "mps" if torch.backends.mps.is_available() else "cpu"
gc.collect()
torch.mps.empty_cache()

# 2. PATHS
BASE_PATH = "/Users/akshatchauhan7/Documents/Documents/Projects/Spectra/PlotQA-Dataset"
DATA_DIR = os.path.join(BASE_PATH, "vbar_categorical_plots")

# 3. LOAD MODEL & PROCESSOR
model_id = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(model_id)
model = VisionEncoderDecoderModel.from_pretrained(model_id)

# ARCHITECTURE SYNC
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_chartqa>'])[0]
model.config.tie_word_embeddings = False 
model.to(device)

# 4. FIX CHECKPOINTERROR: UNFREEZE ENCODER
# Checkpoint logic fails on MPS when layers are frozen
model.encoder.requires_grad_(True)
print("✅ Encoder unfrozen to fix CheckpointError. Adjusting resolution for 16GB RAM safety.")

# 5. DATA LOADING
dataset = load_dataset("imagefolder", data_dir=DATA_DIR)

# 6. DATA COLLATOR (Ultra-Conservative for 16GB)
def collator(batch):
    # Reduced resolution to 384x512 to offset memory of un-frozen encoder
    pixel_values = [processor(item["image"].convert("RGB"), size={"height": 384, "width": 512}, return_tensors="pt").pixel_values for item in batch]
    pixel_values = torch.cat(pixel_values).to(device)
    
    # Process text labels (max_length reduced to 96 for speed/memory)
    labels = [json.loads(item["ground_truth"])["gt_parse"] for item in batch]
    labels_batch = processor.tokenizer(
        labels, padding="max_length", truncation=True, max_length=96, return_tensors="pt"
    ).input_ids.to(device)
    
    labels_batch[labels_batch == processor.tokenizer.pad_token_id] = -100
    
    return {"pixel_values": pixel_values, "labels": labels_batch}

# 7. TRAINING ARGUMENTS (Stable M4 Setup)
training_args = TrainingArguments(
    output_dir="./stem_sight_vbar_final",
    per_device_train_batch_size=1,        
    gradient_accumulation_steps=16,       
    num_train_epochs=3,                   
    learning_rate=2e-5,                   # Lowered for full-model fine-tuning
    logging_steps=10,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,          
    gradient_checkpointing=True,          
    dataloader_num_workers=0,             # Zero workers prevents Segfaults on M4
    report_to="none"
)

# 8. INITIALIZE & TRAIN
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collator,
)

print(f"🚀 Starting VBar Specialist Training on M4...")
trainer.train()

Loading weights: 100%|█| 484/484 [00:00<00:00, 1606.59it/s, Materializing param=encoder.encoder.layers.3.blocks.
The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Encoder unfrozen to fix CheckpointError. Adjusting resolution for 16GB RAM safety.
🚀 Starting VBar Specialist Training on M4...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 